In [1]:
import numpy as np
import pandas as pd
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
from google.colab import drive
from sklearn.metrics import accuracy_score

# Mount Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Load dataset
path = "/content/drive/MyDrive/merged_flight_weather.csv"  # Update with your dataset path
data = pd.read_csv(path)


def time_to_seconds(time_str):
    """Convert time in HHMMSS format to total seconds."""
    try:
        h, m, s = int(time_str[:2]), int(time_str[2:4]), int(time_str[4:])
        return h * 3600 + m * 60 + s
    except ValueError:
        return 0  # Handle invalid time strings

data['CRSDepTime'] = data['CRSDepTime'].astype(str).apply(time_to_seconds)

# Define features and target
features = ['CRSArrTime', 'WindSpeedKmph', 'WindDirDegree', 'WeatherCode',
            'precipMM', 'Visibilty', 'Pressure', 'Cloudcover', 'DewPointF',
            'WindGustKmph', 'tempF', 'WindChillF', 'Humidity', 'Year',
            'Quarter', 'Month', 'DayofMonth', 'CRSDepTime', 'DepDelayMinutes']

target_clf = 'ArrDel15'
data = data.fillna(0)
X_clf = data[features]
y_clf = data[target_clf]

# Split the dataset
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(X_clf, y_clf, test_size=0.3, random_state=42)

from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_clf_train, y_clf_train = sm.fit_resample(X_clf_train, y_clf_train)
X_clf_test, y_clf_test = sm.fit_resample(X_clf_test, y_clf_test)

# Define classifiers
classifiers = {
    "decision_tree": DecisionTreeClassifier(random_state=42),
    "extra_trees": ExtraTreesClassifier(n_estimators=100, random_state=42),
    "logistic_regression": LogisticRegression(max_iter=1000),
    "random_forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "xgboost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

for name, model in classifiers.items():
    model.fit(X_clf_train, y_clf_train)
    y_pred = model.predict(X_clf_test)
    accuracy = accuracy_score(y_clf_test, y_pred)
    print(f"{name} Accuracy: {accuracy}")

print("All classifiers trained and accuracy is displayed.")


decision_tree Accuracy: 0.8996097133048824
extra_trees Accuracy: 0.9023869253957135


/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


logistic_regression Accuracy: 0.8545533074783317
random_forest Accuracy: 0.9040194606255777


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [09:06:51] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


xgboost Accuracy: 0.9205451034013575
All classifiers trained and accuracy is displayed.


In [ ]:
intervals = [(15, 100), (100, 200), (200, 500), (500, 1000), (1000, 2000)]
interval_labels = ['15-100', '100-200', '200-500', '500-1000', '1000-2000']

    # Calculate regression metrics for each interval
metrics = []
for low, high in intervals:
        interval_data = filtered_data[(filtered_data['ArrDelayMinutes'] > low) &
                                      (filtered_data['ArrDelayMinutes'] <= high)]
        if len(interval_data) > 0:
            frequency = len(interval_data)
            X_interval = interval_data[reg_features]
            y_interval = interval_data[reg_target]
            y_pred_interval = reg_pipeline.predict(X_interval)
            interval_mse = mean_squared_error(y_interval, y_pred_interval)
            interval_mae = mean_absolute_error(y_interval, y_pred_interval)
            interval_rmse = np.sqrt(interval_mse)
            interval_r2 = r2_score(y_interval, y_pred_interval)
            metrics.append({
                'Interval': f'{low}-{high}',
                'Frequency' : frequency,
                'MSE': interval_mse,
                'MAE': interval_mae,
                'RMSE': interval_rmse,
                'R2': interval_r2
            })
        else:
            metrics.append({
                'Interval': f'{low}-{high}',
                'Frequency' : None,
                'MSE': None,
                'MAE': None,
                'RMSE': None,
                'R2': None
            })

    # Convert metrics to DataFrame for tabular display
metrics_df = pd.DataFrame(metrics)

    # Print the tabular summary
print("\nMetrics Summary:")
print(metrics_df)